In [5]:
# WEEK 7: NEXT TOKEN PREDICTION

!pip install -q transformers torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load Model
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

print("Loaded Model:", model_name)
print("Vocabulary Size:", tokenizer.vocab_size)


# Function to Predict Next Tokens
def inspect_next_tokens(prompt: str, top_k: int = 10):

    # 1. Convert text into token IDs
    inputs = tokenizer(prompt, return_tensors="pt")

    # 2. Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Extract logits for the last token
    next_token_logits = outputs.logits[0, -1, :]

    # 4. Convert logits into probabilities using Softmax
    probabilities = F.softmax(next_token_logits, dim=-1)

    # 5. Get top-k predictions
    top_probs, top_indices = torch.topk(probabilities, top_k)

    # Display results
    print(f"\nPrompt: '{prompt}'\n")

    print(f"{'Rank':<6} {'Token':<18} {'Token ID':<10} {'Raw Logit':<12} {'Probability':<12}")
    print("-" * 62)

    for rank, (prob, idx) in enumerate(
        zip(top_probs, top_indices), start=1
    ):

        token_str = repr(tokenizer.decode([idx.item()]))
        raw_logit = next_token_logits[idx].item()

        print(
            f"{rank:<6} {token_str:<18} "
            f"{idx.item():<10} {raw_logit:<12.4f} "
            f"{prob.item():.6f}"
        )


# Test the function
inspect_next_tokens("The capital of France is", top_k=10)


# Exercise 1: Factual completion
inspect_next_tokens("Water boils at 100 degrees", top_k=5)

print("\n" + "=" * 62 + "\n")


# Exercise 2: Creative completion
inspect_next_tokens("Once upon a time, there was a", top_k=5)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded Model: distilgpt2
Vocabulary Size: 50257

Prompt: 'The capital of France is'

Rank   Token              Token ID   Raw Logit    Probability 
--------------------------------------------------------------
1      ' the'             262        -57.3821     0.117745
2      ' home'            1363       -57.7406     0.082270
3      ' now'             783        -58.2412     0.049868
4      ' a'               257        -58.2520     0.049335
5      ' being'           852        -58.5779     0.035612
6      ' one'             530        -59.1009     0.021109
7      ' known'           1900       -59.2094     0.018939
8      ' also'            635        -59.2149     0.018834
9      ' in'              287        -59.2623     0.017962
10     ' to'              284        -59.5085     0.014042

Prompt: 'Water boils at 100 degrees'

Rank   Token              Token ID   Raw Logit    Probability 
--------------------------------------------------------------
1      ' Fahrenheit'      35935   